# ST-OMR Meter V5-3A — Robust-Margin Head Candidate

Single fixed TRAIN-only candidate fit pinned to exact CI-green commit `cdc6683a556c16b00e7b154fca8e89ba5dd848b7`. It fits only 2-AI and 3-AI `head.weight` through two preregistered LP stages. Backbone, bias, thresholds, 4-AI, Historical VALIDATION, First-30, V5 VAL, and FINAL_HOLDOUT remain frozen/closed.

In [ ]:
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import time

EXPECTED_HEAD = "cdc6683a556c16b00e7b154fca8e89ba5dd848b7"
EXPECTED_CI_RUN_ID = 32735656612
EXPECTED_SCIPY_VERSION = "1.18.0"
REPOSITORY = "khfy7wpr5p-maker/st-omr-training"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO = Path("/content/st-omr-training")
MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
DATA_ROOT = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
CHECKPOINT_ROOT = MYDRIVE / "ST-OMR-METER-SPECIALISTS"
M4A_ROOT = CHECKPOINT_ROOT / "m4a-234-digit-specialist-dataset-freeze-v2"
D10_ROOT = MYDRIVE / "ST-OMR-D10" / "stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a"
for name, path in {"DATA_ROOT": DATA_ROOT, "CHECKPOINT_ROOT": CHECKPOINT_ROOT, "M4A_ROOT": M4A_ROOT, "D10_ROOT": D10_ROOT}.items():
    if not path.is_dir():
        raise RuntimeError(f"{name} bulunamadi: {path}")
print("DRIVE CHECK = PASS")

if not REPO.exists():
    subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(REPO)])
elif not (REPO / ".git").is_dir():
    raise RuntimeError(f"REPO git repository degil: {REPO}")
remotes = subprocess.check_output(["git", "-C", str(REPO), "remote"], text=True).split()
if "origin" not in remotes:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "add", "origin", REPO_URL])
else:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL])
subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", EXPECTED_HEAD, "--depth", "1"])
fetched_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "FETCH_HEAD"], text=True).strip()
if fetched_head != EXPECTED_HEAD:
    raise RuntimeError(f"FETCH_HEAD mismatch: expected={EXPECTED_HEAD} actual={fetched_head}")
subprocess.check_call(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD])
actual_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual_head != EXPECTED_HEAD:
    raise RuntimeError(f"HEAD mismatch: {actual_head}")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository worktree temiz degil")
print("REPOSITORY CHECK = PASS")
print("HEAD =", actual_head)
print("CI RUN ID =", EXPECTED_CI_RUN_ID)

if metadata.version("scipy") != EXPECTED_SCIPY_VERSION:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "scipy==1.18.0"])
if metadata.version("scipy") != EXPECTED_SCIPY_VERSION:
    raise RuntimeError(f"SciPy version mismatch: {metadata.version('scipy')}")
print("SCIPY VERSION CHECK = PASS", metadata.version("scipy"))

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from st_omr_training import meter_v5_1_bbox_pilot as v51
from st_omr_training import meter_v5_2b_specialist_adaptation as v52b
from st_omr_training import meter_v5_2z_minimum_parameter_change_audit_v1 as v52z
from st_omr_training import meter_v5_3a_robust_margin_head_candidate_v1 as repair
print("MODULE IMPORT = PASS")

DIGIT2_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT2_SHA256)
DIGIT3_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT3_SHA256)
ANN_DIR = DATA_ROOT / "annotations"
V52Z_REPORT = ANN_DIR / v52z.REPORT_NAME
V52Z_ENVELOPE = ANN_DIR / f"v5_2z_execution_envelope_{repair.V52Z_IMPLEMENTATION_HEAD}.json"
REPORT_PATH = ANN_DIR / repair.REPORT_NAME
CANDIDATE_DIR = ANN_DIR / repair.CANDIDATE_DIR_NAME
TEMP_CANDIDATE_DIR = ANN_DIR / repair.TEMP_CANDIDATE_DIR_NAME
ENVELOPE_PATH = ANN_DIR / f"v5_3a_execution_envelope_{EXPECTED_HEAD}.json"
for name, path in {"V5-2Z REPORT": V52Z_REPORT, "V5-2Z ENVELOPE": V52Z_ENVELOPE}.items():
    if not path.is_file():
        raise RuntimeError(f"{name} missing: {path}")
for path in (REPORT_PATH, CANDIDATE_DIR, TEMP_CANDIDATE_DIR, ENVELOPE_PATH):
    if path.exists():
        raise RuntimeError(f"Refusing overwrite/rerun: {path}")
print("EXACT INPUT BINDING = PASS")
print("OUTPUT GUARD = PASS")

required_safety = {
    "single_fixed_candidate_fit": True,
    "linear_head_candidate_fit_authorized": True,
    "candidate_checkpoint_write_authorized": True,
    "candidate_parameter_surface": "head.weight-only-64",
    "frozen_backbone": True,
    "frozen_head_bias": True,
    "autograd_grad_used": False,
    "backward": False,
    "optimizer_steps": 0,
    "automatic_second_configuration": False,
    "hyperparameter_sweep": False,
    "runtime_threshold_tuning": False,
    "alternative_threshold_evaluated": False,
    "new_bbox": False,
    "new_crop_geometry": False,
    "new_spatial_heuristic": False,
    "reserve_v5_train_opened": False,
    "historical_validation_opened": False,
    "historical_retention_executed_by_this_module": False,
    "first30_opened": False,
    "v5_validation_opened": False,
    "final_holdout_locked": True,
    "digit4_frozen": True,
    "production_promotion": False,
}
for key, expected in required_safety.items():
    if repair.safety_boundary().get(key) != expected:
        raise RuntimeError(f"Safety boundary mismatch: {key}")
solver = repair.solver_contract()
if solver.get("library_version_matches_expected") is not True:
    raise RuntimeError("Pinned SciPy solver contract mismatch")
if solver.get("robust_decision_margin") != 0.25:
    raise RuntimeError("Robust margin contract changed")
if solver.get("primary_objective") != "minimum_total_absolute_delta_weight_l1":
    raise RuntimeError("Primary L1 objective changed")
if solver.get("secondary_objective") != "minimum_max_absolute_delta_weight_linf":
    raise RuntimeError("Secondary Linf objective changed")
if solver.get("primary_objective_fixed_before_secondary") is not True:
    raise RuntimeError("Lexicographic order changed")
if solver.get("automatic_second_configuration") is not False or solver.get("solver_sweep") is not False:
    raise RuntimeError("Fallback configuration or sweep enabled")
if solver.get("threshold_search") is not False or solver.get("bias_search") is not False:
    raise RuntimeError("Threshold or bias search enabled")
print("V5-3A CONTRACT = PASS")
print("CANDIDATE SURFACE = HEAD.WEIGHT ONLY")
print("PRIMARY=L1 | SECONDARY=LINF | ROBUST_MARGIN=0.25")
print("AUTOGRAD=False | BACKWARD=False | OPTIMIZER_STEPS=0")
print("THRESHOLDS=FROZEN | BIAS=FROZEN | BACKBONE=FROZEN | 4-AI=FROZEN")
print("HISTORICAL_VALIDATION=CLOSED | FIRST-30=CLOSED")
print("V5_VAL=CLOSED | FINAL_HOLDOUT=LOCKED")

started = time.time()
def progress(processed, total, phase):
    if processed == 1 or processed == total or processed % 2048 == 0:
        print(phase, f"{processed}/{total}", f"| elapsed={int(time.time() - started)}s")

report = repair.fit_robust_margin_head_candidates_v1(
    DATA_ROOT,
    m4a_root=M4A_ROOT,
    d10_root=D10_ROOT,
    digit2_frozen=DIGIT2_FROZEN,
    digit3_frozen=DIGIT3_FROZEN,
    v5_2z_report=V52Z_REPORT,
    v5_2z_envelope=V52Z_ENVELOPE,
    confirmation=repair.APPROVAL_TOKEN,
    progress=progress,
)
if not REPORT_PATH.is_file():
    raise RuntimeError(f"Candidate report missing: {REPORT_PATH}")
report_bytes = REPORT_PATH.read_bytes()
saved_report = json.loads(report_bytes.decode("utf-8"))
if saved_report != report:
    raise RuntimeError("Saved report mismatch")
for key, expected in required_safety.items():
    if report.get(key) != expected:
        raise RuntimeError(f"Saved report safety mismatch: {key}")
gate = report.get("candidate_selection_gate")
if gate not in {"PASS", "HOLD"}:
    raise RuntimeError(f"Unknown candidate gate: {gate}")
if gate == "PASS":
    if report.get("candidate_checkpoint_written") is not True:
        raise RuntimeError("PASS without candidate checkpoint write")
    if not CANDIDATE_DIR.is_dir() or TEMP_CANDIDATE_DIR.exists():
        raise RuntimeError("Candidate directory transaction incomplete")
    for digit in ("2", "3"):
        specialist = report["per_specialist"][digit]
        fit = specialist["fit"]
        if fit.get("candidate_claim") != "CANDIDATE_WITNESS_VERIFIED":
            raise RuntimeError(f"{digit}-AI candidate witness not verified")
        if fit.get("robust_candidate_witness_verified") is not True:
            raise RuntimeError(f"{digit}-AI witness flag mismatch")
        if fit.get("primary_optimality_claim") != "SOLVER_REPORTED_OPTIMAL_NOT_FORMAL_PROOF":
            raise RuntimeError(f"{digit}-AI primary optimality evidence changed")
        if fit.get("secondary_optimality_claim") != "SOLVER_REPORTED_OPTIMAL_NOT_FORMAL_PROOF":
            raise RuntimeError(f"{digit}-AI secondary optimality evidence changed")
        for key in ("primary_decision_constraint_violations", "primary_auxiliary_bound_violations", "primary_l1_cap_violations", "primary_lower_bound_conflicts", "parameter_bound_violations", "v5_constraint_violations", "v5_solver_margin_constraint_violations", "historical_margin_constraint_violations", "historical_solver_margin_constraint_violations"):
            if fit.get(key) != 0:
                raise RuntimeError(f"{digit}-AI residual violation: {key}")
        if fit.get("functional_delta_identity_verified") is not True:
            raise RuntimeError(f"{digit}-AI functional identity failed")
        if fit["diagnostic_v5_train_metrics"].get("f1") != 1.0:
            raise RuntimeError(f"{digit}-AI V5 TRAIN F1 is not 1.0")
        if fit["historical_transition_counts"].get("correct_to_wrong") != 0:
            raise RuntimeError(f"{digit}-AI frozen-correct historical regression")
        if specialist["float32_copy_gate"].get("gate") != "PASS":
            raise RuntimeError(f"{digit}-AI independent float32 gate HOLD")
        if specialist["runtime_float32_gate"].get("gate") != "PASS":
            raise RuntimeError(f"{digit}-AI actual runtime float32 gate HOLD")
        invariants = specialist["state_invariants"]
        for key in ("only_head_weight_changed", "backbone_bit_identical", "head_bias_bit_identical"):
            if invariants.get(key) is not True:
                raise RuntimeError(f"{digit}-AI state invariant failed: {key}")
        candidate = specialist["candidate"]
        if candidate.get("reload_verified") is not True:
            raise RuntimeError(f"{digit}-AI candidate reload not verified")
        candidate_path = Path(candidate["candidate_path"])
        if candidate_path.parent != CANDIDATE_DIR or not candidate_path.is_file():
            raise RuntimeError(f"{digit}-AI candidate path mismatch")
        candidate_sha = hashlib.sha256(candidate_path.read_bytes()).hexdigest()
        if candidate_sha != candidate.get("candidate_sha256"):
            raise RuntimeError(f"{digit}-AI candidate SHA mismatch")
elif CANDIDATE_DIR.exists() or report.get("candidate_checkpoint_written") is not False:
    raise RuntimeError("HOLD published a candidate artifact")

post_run_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if post_run_head != EXPECTED_HEAD:
    raise RuntimeError(f"Post-run HEAD mismatch: {post_run_head}")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository changed during candidate fit")
report_sha256 = hashlib.sha256(report_bytes).hexdigest()
candidate_hashes = {}
if gate == "PASS":
    candidate_hashes = {digit: report["per_specialist"][digit]["candidate"]["candidate_sha256"] for digit in ("2", "3")}
envelope = {
    "schema": "st-omr-meter-v5-3a-exact-sha-execution-envelope-v1",
    "repository": REPOSITORY,
    "expected_head": EXPECTED_HEAD,
    "actual_head_before_run": actual_head,
    "actual_head_after_run": post_run_head,
    "ci_run_id": EXPECTED_CI_RUN_ID,
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "report_name": repair.REPORT_NAME,
    "report_sha256": report_sha256,
    "candidate_selection_gate": gate,
    "candidate_sha256": candidate_hashes,
    "solver_contract": report["solver_contract"],
    "claims": {digit: report["per_specialist"][digit]["fit"]["candidate_claim"] for digit in ("2", "3")},
    "safety_boundary": {key: report[key] for key in required_safety},
}
v51._atomic_write_json(ENVELOPE_PATH, envelope)
envelope_sha256 = hashlib.sha256(ENVELOPE_PATH.read_bytes()).hexdigest()

print()
print("============================================")
print("V5-3A ROBUST-MARGIN HEAD CANDIDATE RESULT")
print("============================================")
print("CANDIDATE SELECTION GATE =", gate)
for digit in ("2", "3"):
    specialist = report["per_specialist"][digit]
    fit = specialist["fit"]
    print()
    print(f"========== {digit}-AI ==========")
    print("CLAIM =", fit["candidate_claim"])
    print("PRIMARY SOLVER =", {key: fit.get(key) for key in ("primary_status", "primary_success", "primary_iterations", "primary_optimality_claim")})
    print("SECONDARY SOLVER =", {key: fit.get(key) for key in ("secondary_status", "secondary_success", "secondary_iterations", "secondary_optimality_claim")})
    if fit.get("candidate_claim") == "CANDIDATE_WITNESS_VERIFIED":
        print("MINIMUM DELTA WEIGHT L1 =", fit["minimum_delta_weight_l1"])
        print("MINIMUM DELTA WEIGHT LINF =", fit["minimum_delta_weight_linf"])
        print("MINIMUM V5 MARGIN =", fit["minimum_v5_signed_decision_margin"])
        print("MINIMUM HISTORICAL RETAINED MARGIN =", fit["minimum_historical_retained_signed_decision_margin"])
        print("V5 TRAIN METRICS =", fit["diagnostic_v5_train_metrics"])
        print("HISTORICAL TRAIN METRICS =", fit["diagnostic_historical_train_metrics"])
        print("HISTORICAL TRANSITIONS =", fit["historical_transition_counts"])
        print("HISTORICAL ABS LOGIT DRIFT =", fit["historical_absolute_logit_drift"])
        print("WEIGHT GEOMETRY =", fit["weight_geometry"])
        print("FLOAT32 COPY GATE =", specialist["float32_copy_gate"])
        print("RUNTIME FLOAT32 GATE =", specialist["runtime_float32_gate"])
        print("STATE INTEGRITY =", specialist["state_invariants"])
        print("CANDIDATE =", specialist.get("candidate"))
    print("PATH DIAGNOSIS =", specialist["path_diagnosis"])
print()
print("EXACT SHA EXECUTION = PASS")
print("HEAD =", post_run_head)
print("REPORT =", REPORT_PATH)
print("REPORT SHA256 =", report_sha256)
print("EXECUTION ENVELOPE =", ENVELOPE_PATH)
print("ENVELOPE SHA256 =", envelope_sha256)
print("LINEAR PROGRAM CANDIDATE FIT EXECUTED = True")
print("GRADIENT TRAINING = False | AUTOGRAD = False | BACKWARD = False")
print("HISTORICAL RETENTION EXECUTED = False")
print("FIRST-30 = CLOSED | V5 VAL = CLOSED | FINAL HOLDOUT = LOCKED")
